In [1]:
import os
import numpy as np
import decord
import imageio
import tempfile
from PIL import Image
from IPython.display import Video, display

def create_temp_video(pil_images, fps=4):
    with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as temp:
        writer = imageio.get_writer(temp.name, fps=fps)

        for img in pil_images:
            frame = np.array(img)  # Convert PIL.Image to numpy array
            writer.append_data(frame)

        writer.close()
        return temp.name

/home/andy/miniconda3/envs/matrix/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2025-04-15 14:11:57,350] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/home/andy/miniconda3/envs/matrix/lib/python3.10/site-packages/torch/utils/cpp_extension.py:1964: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(


In [ ]:
from vlm_reward_model import OpenAIRewardModel

gpt4_rm = OpenAIRewardModel(
    model="gpt-4o",
    api_key=os.environ["OPENAI_API_KEY"],
    reward_query = (
        "You are a video analyst. Your task is to analyze a sequence of consecutive images and describe the spatial relationship "
        "between the car and any potential obstacles. Based on this analysis, assess the risk of a possible collision. "
    ),
    reward_criteria = (
        "Here is the video description: {} "
        "Return 1 if there is no risk of collision with any obstacle. "
        "Return 0 if there is a potential risk of collision, but no collision has occurred yet. "
        "Return -1 if you believe a collision has already occurred between the car and an obstacle, regardless of whether there was damage."
        "Your response must only contain one of the following: 1, 0, or -1. Do not include any additional explanation or description."
    ),
)


In [2]:
video_path = "/home/andy/matrix/wm_gym/example_video/hit_tree_example.mp4"
video_reader = decord.VideoReader(video_path)
num_frames = len(video_reader)
all_frames = video_reader.get_batch(list(range(num_frames))).asnumpy()
# Convert to PIL images
pil_frames = [Image.fromarray(frame) for frame in all_frames]

In [3]:
i = 50
drive_normal_clip = pil_frames[i: i+4]
video_path = create_temp_video(drive_normal_clip)
display(Video(video_path, embed=True))
full_response_pil, short_answer_pil = gpt4_rm.analyze(drive_normal_clip)
print("Full response (PIL):", full_response_pil)
print("Extracted answer (PIL):", short_answer_pil)

Full response (PIL): The sequence of images shows a car driving on a dirt path in an open landscape. The car is moving away from the viewer, and the environment consists of sparse vegetation and open terrain. 

### Spatial Relationship:
- **Car Position**: The car is centered in the frame, maintaining a consistent path.
- **Obstacles**: There are no immediate obstacles directly in the car's path. The vegetation is scattered and appears to be at a safe distance from the car's trajectory.

### Risk Assessment:
- **Collision Risk**: Low. The car is on a clear path with no visible obstacles in its immediate vicinity. The open terrain provides ample space for maneuvering if needed.

Overall, the car is in a safe position with minimal risk of collision based on the current images.
Extracted answer (PIL): 1


In [7]:
i = 160
crash_tree_clip = pil_frames[i: i+4]
video_path = create_temp_video(crash_tree_clip)
display(Video(video_path, embed=True))
full_response_pil, short_answer_pil = gpt4_rm.analyze(crash_tree_clip)
print("Full response (PIL):", full_response_pil)
print("Extracted answer (PIL):", short_answer_pil)

Full response (PIL): The sequence of images shows a white car with a large rear spoiler moving towards a group of trees. Here's the analysis:

1. **Image 1**: The car is approaching the trees but still has some distance. The shadow indicates the car is moving into a shaded area.

2. **Image 2**: The car is closer to the trees. The gap between the car and the trees has decreased, suggesting forward movement.

3. **Image 3**: The car is very close to the trees. The shadow and lighting suggest minimal space between the car and the trees.

4. **Image 4**: The car appears to be almost touching the trees. The proximity indicates a high risk of collision if the car continues on this path.

**Risk Assessment**: The risk of collision is high as the car is moving directly towards the trees with little space left. Immediate action is needed to avoid a collision, such as steering away or stopping.
Extracted answer (PIL): 0
